# Extract Driving Test Questions from PDF

This notebook parses the PDF containing 250 multiple-choice driving test questions and converts them into the required JSON structure.

In [2]:
import json
import re
from pathlib import Path

pdf_path = Path('assets/questions/Tài liệu_Bộ 250 câu hỏi hạng A1,A.pdf')
json_template_path = Path('assets/questions/questions_a1.json')
print('PDF exists:', pdf_path.exists())
print('Template exists:', json_template_path.exists())

PDF exists: True
Template exists: True


In [3]:
import subprocess
import sys

try:
    import fitz
except ImportError:
    print('Installing PyMuPDF...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pymupdf'])
    import fitz

print('fitz version:', fitz.__doc__ if hasattr(fitz, '__doc__') else 'unknown')
print('PDF path exists:', pdf_path.exists())

fitz version: PyMuPDF 1.27.2.3: Python bindings for the MuPDF 1.27.2 library.
Python 3.13 running on win32 (64-bit).

PDF path exists: True


In [12]:
doc = fitz.open(pdf_path)
print('Total pages:', len(doc))

page = doc[0]
page_dict = page.get_text('dict')
print('Page dict keys:', page_dict.keys())
print('Blocks on first page:', len(page_dict['blocks']))
for i, block in enumerate(page_dict['blocks'][:3], 1):
    print('\nBlock', i, 'type', block['type'], 'bbox', block['bbox'])
    if block['type'] == 0:
        for line in block['lines'][:3]:
            text = ''.join(span['text'] for span in line['spans'])
            print('  LINE:', text)
            for span in line['spans']:
                print('    span:', repr(span['text']), 'flags', span.get('flags'), 'color', span.get('color'), 'font', span.get('font'))

Total pages: 74
Page dict keys: dict_keys(['width', 'height', 'blocks'])
Blocks on first page: 10

Block 1 type 1 bbox (251.0399932861328, 138.36001586914062, 371.760009765625, 287.52001953125)

Block 2 type 1 bbox (84.95999908447266, 511.4400634765625, 538.5599975585938, 760.9200439453125)

Block 3 type 0 bbox (78.02400207519531, 36.090396881103516, 319.05999755859375, 69.03263854980469)
  LINE: 1 
    span: '1 ' flags 4 color 0 font Times New Roman
  LINE:  
    span: ' ' flags 4 color 0 font Times New Roman


In [13]:
all_text = ''
for page_num in range(len(doc)):
    page = doc[page_num]
    text = page.get_text()
    all_text += f'--- Page {page_num + 1} ---\n{text}\n'

print('Total text length:', len(all_text))
print('First 2000 chars:')
print(all_text[:2000])

Total text length: 69830
First 2000 chars:
--- Page 1 ---
1 
 
BỘ CÔNG AN 
CỤC CẢNH SÁT GIAO THÔNG 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
250 CÂU HỎI 
DÙNG CHO SÁT HẠCH LÁI XE 
CƠ GIỚI ĐƯỜNG BỘ 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Hà Nội - 2025 

--- Page 2 ---
LỜI NÓI ĐẦU 
 
An toàn giao thông là nền tảng bảo đảm cho sự phát triển bền vững của xã 
hội. Trong đó, việc đào tạo, sát hạch và cấp giấy phép lái xe cơ giới đường bộ 
giữ vai trò đặc biệt quan trọng nhằm hình thành đội ngũ lái xe có đạo đức, trình 
độ chuyên môn và ý thức trách nhiệm cao. 
Thực hiện chỉ đạo của Chính phủ và Bộ Công an về việc nâng cao chất 
lượng sát hạch lái xe, Cục Cảnh sát giao thông tổ chức biên soạn Bộ 250 câu hỏi 
dùng cho sát hạch lái xe cơ giới đường bộ. Bộ câu hỏi được xây dựng khoa học, 
cập nhật các quy định mới nhất của pháp luật, bảo đảm phù hợp với yêu cầu thực 
tiễn và sát với năng lực cần có của người lái xe. 
Bộ câu hỏi được bố cục thành 5 chương, bao gồm: 
1. Chương I. Quy định chung và quy tắc gia

In [14]:
questions_raw = re.findall(r'Câu \d+\..*?(?=Câu \d+\.|$)', all_text, re.DOTALL)
print('Number of questions found:', len(questions_raw))
for i, q in enumerate(questions_raw[:3], 1):
    print(f'\n--- Question {i} ---')
    print(repr(q[:500]))

Number of questions found: 249

--- Question 1 ---
'Câu 1. Phần của đường bộ được sử dụng cho phương tiện giao thông đường \nbộ đi lại là gì? \n1. Phần mặt đường và lề đường. \n2. Phần đường xe chạy. \n3. Phần đường xe cơ giới. \n'

--- Question 2 ---
'Câu 2. Làn đường là gì? \n1. Là một phần của phần đường xe chạy được chia theo chiều dọc của đường, sử \ndụng cho xe chạy. \n2. Là một phần của phần đường xe chạy được chia theo chiều dọc của đường, có \nđủ chiều rộng cho xe chạy an toàn. \n3. Là đường cho xe ô tô chạy, dừng, đỗ an toàn. \n'

--- Question 3 ---
'Câu 3. Khổ giới hạn của đường bộ được hiểu như thế nào là đúng? \n1. Khổ giới hạn của đường bộ là khoảng trống có kích thước giới hạn về chiều \nrộng, chiều cao của đường bộ để các xe, bao gồm cả hàng hoá xếp trên xe đi qua \nđược an toàn và được xác định theo quy chuẩn, tiêu chuẩn kỹ thuật của đường bộ. \n2. Là khoảng trống có kích thước giới hạn về chiều rộng của đường, cầu, bến phà, \nhầm trên đường bộ để các xe kể cả hàng hóa

In [15]:
underlined_texts = []
red_texts = []

for page_num in range(len(doc)):
    page = doc[page_num]
    page_dict = page.get_text('dict')
    for block in page_dict['blocks']:
        if block['type'] == 0:  # text block
            for line in block['lines']:
                for span in line['spans']:
                    text = span['text'].strip()
                    if not text:
                        continue
                    flags = span.get('flags', 0)
                    color = span.get('color', 0)
                    if flags & 4:  # underline
                        underlined_texts.append(text)
                    if color == (1, 0, 0, 1) or (isinstance(color, int) and color == 1):  # red
                        red_texts.append(text)

print('Underlined texts count:', len(underlined_texts))
print('Sample underlined:', underlined_texts[:10])
print('Red texts count:', len(red_texts))
print('Sample red:', red_texts[:10])

Underlined texts count: 2302
Sample underlined: ['1', 'BỘ CÔNG AN', 'CỤC CẢNH SÁT GIAO THÔNG', '250 CÂU HỎI', 'DÙNG CHO SÁT HẠCH LÁI XE', 'CƠ GIỚI ĐƯỜNG BỘ', 'Hà Nội - 2025', 'LỜI NÓI ĐẦU', 'An toàn giao thông là nền tảng bảo đảm cho sự phát triển bền vững của xã', 'hội. Trong đó, việc đào tạo, sát hạch và cấp giấy phép lái xe cơ giới đường bộ']
Red texts count: 0
Sample red: []


In [16]:
def extract_formatted_text(page):
    formatted = ''
    page_dict = page.get_text('dict')
    for block in page_dict['blocks']:
        if block['type'] == 0:  # text
            for line in block['lines']:
                line_text = ''
                for span in line['spans']:
                    text = span['text']
                    flags = span.get('flags', 0)
                    if flags & 4:  # underline
                        text = f'<u>{text}</u>'
                    line_text += text
                formatted += line_text + '\n'
        elif block['type'] == 1:  # image
            formatted += '[IMAGE]\n'
    return formatted

all_formatted_text = ''
for page_num in range(len(doc)):
    page = doc[page_num]
    formatted = extract_formatted_text(page)
    all_formatted_text += f'--- Page {page_num + 1} ---\n{formatted}\n'

print('Formatted text length:', len(all_formatted_text))
print('First 3000 chars:')
print(all_formatted_text[:3000])

Formatted text length: 89197
First 3000 chars:
--- Page 1 ---
[IMAGE]
[IMAGE]
<u>1 </u>
<u> </u>
<u>BỘ CÔNG AN </u>
<u>CỤC CẢNH SÁT GIAO THÔNG </u>
<u> </u>
<u> </u>
<u> </u>
<u> </u>
<u> </u>
<u> </u>
<u> </u>
<u> </u>
<u> </u>
<u> </u>
<u> </u>
<u> </u>
<u> </u>
<u> </u>
<u> </u>
<u> </u>
<u>250 CÂU HỎI </u>
<u>DÙNG CHO SÁT HẠCH LÁI XE </u>
<u>CƠ GIỚI ĐƯỜNG BỘ </u>
<u> </u>
<u> </u>
<u> </u>
<u> </u>
<u> </u>
<u> </u>
<u> </u>
<u> </u>
<u> </u>
<u> </u>
<u> </u>
<u> </u>
<u> </u>
<u> </u>
<u> </u>
<u> </u>
<u> </u>
<u> </u>
<u>Hà Nội - 2025 </u>

--- Page 2 ---
<u>LỜI NÓI ĐẦU </u>
<u> </u>
<u>An toàn giao thông là nền tảng bảo đảm cho sự phát triển bền vững của xã </u>
<u>hội. Trong đó, việc đào tạo, sát hạch và cấp giấy phép lái xe cơ giới đường bộ </u>
<u>giữ vai trò đặc biệt quan trọng nhằm hình thành đội ngũ lái xe có đạo đức, trình </u>
<u>độ chuyên môn và ý thức trách nhiệm cao. </u>
<u>Thực hiện chỉ đạo của Chính phủ và Bộ Công an về việc nâng cao chất </u>
<u>lượng sát hạch l

In [21]:
    # Remove <u> tags from content
    content = re.sub(r'</?u>', '', content).replace('\n', ' ').strip().replace('\n', ' ').strip()

NameError: name 'content' is not defined

In [35]:
import json
import re
from pathlib import Path
import fitz

pdf_path = Path('assets/questions/Tài liệu_Bộ 250 câu hỏi hạng A1,A.pdf')

def get_chapter(qid):
    if 1 <= qid <= 100:
        return "Quy Định Chung Và Quy Tắc Giao Thông Đường Bộ"
    elif 101 <= qid <= 110:
        return "Văn Hóa Giao Thông Và Đạo Đức Người Lái Xe"
    elif 111 <= qid <= 125:
        return "Kỹ Thuật Lái Xe"
    elif 126 <= qid <= 215:
        return "Biển Báo Đường Bộ"
    elif 216 <= qid <= 250:
        return "Sa Hình Và Tình Huống Giao Thông"
    else:
        return ""

def extract_questions(pdf_path):
    doc = fitz.open(pdf_path)
    questions = []
    
    # Get all text
    full_text = ""
    for page in doc:
        full_text += page.get_text()
    
    # Split into questions
    question_blocks = re.split(r'(?=Câu \d+\.)', full_text)
    
    for block in question_blocks[1:]:  # Skip the first empty
        # Extract id
        match = re.match(r'Câu (\d+)\.\s*(.*)', block, re.DOTALL)
        if not match:
            continue
        qid = int(match.group(1))
        q_text = match.group(2).strip()
        
        # Find answers
        answers = []
        correct = -1
        content = ""
        
        # Split by lines
        lines = q_text.split('\n')
        i = 0
        while i < len(lines):
            line = lines[i].strip()
            if not line:
                i += 1
                continue
            ans_match = re.match(r'(\d+)\.\s*(.*)', line)
            if ans_match:
                num = int(ans_match.group(1))
                text = ans_match.group(2).strip()
                if num <= 4:
                    answers.append(text)
                    # Check if this line has underline, but since plain text, can't
                    # For now, assume we need to use the dict
                else:
                    content += line + " "
            else:
                content += line + " "
            i += 1
        
        # For correct, since plain text, we need to use the formatted text
        # Let's get the formatted block
        formatted_block = ""
        # But to simplify, since we have the formatted text, let's use that
        
        # Earlier we have all_formatted_text
        # Let's find the corresponding block in all_formatted_text
        
        # For now, since the correct is underlined, and in plain text it's not, but in the earlier test, we detected it.
        
        # To make it work, let's modify to use the formatted text for detection.
        
        # Let's re-do the extraction using formatted text.
        
        # Get formatted text
        formatted_text = ""
        for page_num in range(len(doc)):
            page = doc[page_num]
            page_dict = page.get_text('dict')
            for block in page_dict['blocks']:
                if block['type'] == 0:
                    for line in block['lines']:
                        line_text = ''
                        for span in line['spans']:
                            text = span['text']
                            flags = span.get('flags', 0)
                            if flags & 4:
                                text = f'<u>{text}</u>'
                            line_text += text
                        formatted_text += line_text + '\n'
                elif block['type'] == 1:
                    formatted_text += '[IMAGE]\n'
        
        # Split formatted questions
        formatted_questions = re.split(r'(?=Câu \d+\.)', formatted_text)[1:]
        print(f'Number of formatted questions: {len(formatted_questions)}')
        
        for fq in formatted_questions:
            match = re.match(r'Câu (\d+)\.\s*(.*)', fq, re.DOTALL)
            if not match:
                continue
            qid = int(match.group(1))
            q_text = match.group(2).strip()
            
            # Find content and answers
            match1 = re.search(r'\b1\.', q_text)
            if match1:
                content = q_text[:match1.start()].strip()
                answers_part = q_text[match1.start():]
            else:
                content = q_text
                answers_part = ''
            
            # Remove <u> from content
            content = re.sub(r'</?u>', '', content).replace('\n', ' ').strip()
            
            # Extract answers
            answers = []
            for i in range(1, 5):
                pattern = rf'\b{i}\.\s*(.*?)(?=\b{i+1}\.|$)'
                match = re.search(pattern, answers_part, re.DOTALL)
                if match:
                    ans = match.group(1).strip()
                    ans = re.sub(r'</?u>', '', ans).replace('\n', ' ').strip()
                    answers.append(ans)
            
            # Find correct
            correct = -1
            for i in range(1, 5):
                if f'<u>{i}.</u>' in answers_part:
                    correct = i - 1
                    break
            
            # Chapter
            chapter = get_chapter(qid)
            
            # Important: false
            is_important = False
            
            # Image
            has_image = '[IMAGE]' in fq
            image = None if has_image else None
            
            questions.append({
                "id": qid,
                "chapter": chapter,
                "content": content,
                "answers": answers,
                "correctAnswer": correct,
                "explanation": "",
                "isImportant": is_important,
                "image": image
            })
    
    print(f'Inside function: {len(questions)} questions')
    doc.close()
    return questions

questions = extract_questions(pdf_path)
print(f'Extracted {len(questions)} questions')

Number of formatted questions: 249
Number of formatted questions: 249
Number of formatted questions: 249
Number of formatted questions: 249
Number of formatted questions: 249
Number of formatted questions: 249
Number of formatted questions: 249
Number of formatted questions: 249
Number of formatted questions: 249
Number of formatted questions: 249
Number of formatted questions: 249
Number of formatted questions: 249
Number of formatted questions: 249
Number of formatted questions: 249
Number of formatted questions: 249
Number of formatted questions: 249
Number of formatted questions: 249
Number of formatted questions: 249
Number of formatted questions: 249
Number of formatted questions: 249
Number of formatted questions: 249
Number of formatted questions: 249
Number of formatted questions: 249
Number of formatted questions: 249
Number of formatted questions: 249
Number of formatted questions: 249
Number of formatted questions: 249
Number of formatted questions: 249
Number of formatted 

In [25]:
# Print full text for first few questions
doc = fitz.open(pdf_path)
full_text = ""
for page in doc:
    full_text += page.get_text()

question_blocks = re.split(r'(?=Câu \d+\.)', full_text)
for i, block in enumerate(question_blocks[:6]):
    print(f"Block {i}:")
    print(repr(block[:500]))  # First 500 chars
    print()

doc.close()

Block 0:
'1 \n \nBỘ CÔNG AN \nCỤC CẢNH SÁT GIAO THÔNG \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n250 CÂU HỎI \nDÙNG CHO SÁT HẠCH LÁI XE \nCƠ GIỚI ĐƯỜNG BỘ \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nHà Nội - 2025 \nLỜI NÓI ĐẦU \n \nAn toàn giao thông là nền tảng bảo đảm cho sự phát triển bền vững của xã \nhội. Trong đó, việc đào tạo, sát hạch và cấp giấy phép lái xe cơ giới đường bộ \ngiữ vai trò đặc biệt quan trọng nhằm hình thành đội ngũ lái xe có đạo đức, trình \nđộ chuyên môn và ý thức trách nhiệm cao. \nThực hiện chỉ đạo của Chí'

Block 1:
'Câu 1. Phần của đường bộ được sử dụng cho phương tiện giao thông đường \nbộ đi lại là gì? \n1. Phần mặt đường và lề đường. \n2. Phần đường xe chạy. \n3. Phần đường xe cơ giới. \n'

Block 2:
'Câu 2. Làn đường là gì? \n1. Là một phần của phần đường xe chạy được chia theo chiều dọc của đường, sử \ndụng cho xe chạy. \n2. Là một phần của phần đường xe chạy được chia theo chiều dọc của đường, có \nđủ chiều rộng cho xe chạy an toàn.

In [32]:
# Check extracted questions
with open('assets/questions/questions_a1_extracted.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

ids = [q['id'] for q in data]
print('Total questions:', len(ids))
print('IDs:', sorted(ids))
missing = [i for i in range(1,251) if i not in ids]
print('Missing IDs:', missing)

# Check if all have 4 answers
for q in data:
    if len(q['answers']) != 4:
        print(f"Question {q['id']} has {len(q['answers'])} answers")
    if q['correctAnswer'] == -1:
        print(f"Question {q['id']} has no correct answer")

Total questions: 62001
IDs: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,

In [33]:
# Check unique ids
ids = [q['id'] for q in data]
unique_ids = set(ids)
print('Total entries:', len(ids))
print('Unique IDs:', len(unique_ids))
print('Max ID:', max(unique_ids) if unique_ids else 0)
print('Min ID:', min(unique_ids) if unique_ids else 0)

# Filter to 1-250
filtered = [q for q in data if 1 <= q['id'] <= 250]
print('Filtered to 1-250:', len(filtered))

# Sort and save
filtered.sort(key=lambda x: x['id'])
with open('assets/questions/questions_a1_final.json', 'w', encoding='utf-8') as f:
    json.dump(filtered, f, ensure_ascii=False, indent=2)

print('Saved filtered to questions_a1_final.json')

Total entries: 62001
Unique IDs: 249
Max ID: 250
Min ID: 1
Filtered to 1-250: 62001
Saved filtered to questions_a1_final.json


In [36]:
# Load final, remove duplicates
with open('assets/questions/questions_a1_final.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

seen = set()
unique_questions = []
for q in data:
    qid = q['id']
    if qid not in seen:
        seen.add(qid)
        unique_questions.append(q)

unique_questions.sort(key=lambda x: x['id'])

print(f'Unique questions: {len(unique_questions)}')

# Check if all have 4 answers
for q in unique_questions:
    if len(q['answers']) != 4:
        print(f"Question {q['id']} has {len(q['answers'])} answers: {q['answers']}")
    if q['correctAnswer'] == -1:
        print(f"Question {q['id']} has no correct answer")

# Save unique
with open('assets/questions/questions_a1_unique.json', 'w', encoding='utf-8') as f:
    json.dump(unique_questions, f, ensure_ascii=False, indent=2)

print('Saved unique to questions_a1_unique.json')

Unique questions: 249
Question 1 has 3 answers: ['Phần mặt đường và lề đường.', 'Phần đường xe chạy.', 'Phần đường xe cơ giới.']
Question 2 has 3 answers: ['Là một phần của phần đường xe chạy được chia theo chiều dọc của đường, sử  dụng cho xe chạy.', 'Là một phần của phần đường xe chạy được chia theo chiều dọc của đường, có  đủ chiều rộng cho xe chạy an toàn.', 'Là đường cho xe ô tô chạy, dừng, đỗ an toàn.']
Question 3 has 3 answers: ['Khổ giới hạn của đường bộ là khoảng trống có kích thước giới hạn về chiều  rộng, chiều cao của đường bộ để các xe, bao gồm cả hàng hoá xếp trên xe đi qua  được an toàn và được xác định theo quy chuẩn, tiêu chuẩn kỹ thuật của đường bộ.', 'Là khoảng trống có kích thước giới hạn về chiều rộng của đường, cầu, bến phà,  hầm trên đường bộ để các xe kể cả hàng hóa xếp trên xe đi qua được an toàn.', 'Là khoảng trống có kích thước giới hạn về chiều cao của cầu, bến phà, hầm  trên đường bộ để các xe đi qua được an toàn.']
Question 4 has 3 answers: ['Để phân chia 

In [30]:
# Debug formatted_questions
import fitz
import re

pdf_path = Path('assets/questions/Tài liệu_Bộ 250 câu hỏi hạng A1,A.pdf')
doc = fitz.open(pdf_path)
formatted_text = ""
for page_num in range(len(doc)):
    page = doc[page_num]
    page_dict = page.get_text('dict')
    for block in page_dict['blocks']:
        if block['type'] == 0:
            for line in block['lines']:
                line_text = ''
                for span in line['spans']:
                    text = span['text']
                    flags = span.get('flags', 0)
                    if flags & 4:
                        text = f'<u>{text}</u>'
                    line_text += text
                formatted_text += line_text + '\n'
        elif block['type'] == 1:
            formatted_text += '[IMAGE]\n'

formatted_questions = re.split(r'(?=Câu \d+\.)', formatted_text)[1:]
print(f'Number of formatted questions: {len(formatted_questions)}')
for i in range(min(5, len(formatted_questions))):
    print(f'Question {i}: {repr(formatted_questions[i][:200])}')

doc.close()

Number of formatted questions: 249
Question 0: 'Câu 1. Phần của đường bộ được sử dụng cho phương tiện giao thông đường </u>\n<u>bộ đi lại là gì? </u>\n<u>1.</u> <u>Phần mặt đường và lề đường. </u>\n<u>2.</u> <u>Phần đường xe chạy. </u>\n<u>3.</u> <u>Ph'
Question 1: 'Câu 2. Làn đường là gì? </u>\n<u>1.</u> <u>Là một phần của phần đường xe chạy được chia theo chiều dọc của đường, sử </u>\n<u>dụng cho xe chạy. </u>\n<u>2.</u> <u>Là một phần của phần đường xe chạy được '
Question 2: 'Câu 3. Khổ giới hạn của đường bộ được hiểu như thế nào là đúng? </u>\n<u>1.</u> <u>Khổ giới hạn của đường bộ là khoảng trống có kích thước giới hạn về chiều </u>\n<u>rộng, chiều cao của đường bộ để các '
Question 3: 'Câu 4. Dải phân cách được lắp đặt để làm gì? </u>\n<u>1.</u> <u>Để phân chia các làn đường dành cho xe cơ giới và xe thô sơ trên đường cao </u>\n<u>tốc. </u>\n<u>2.</u> <u>Để phân chia phần đường xe chạy'
Question 4: 'Câu 5. Vạch kẻ đường là gì? </u>\n<u>1.</u> <u>Là báo hiệu đường bộ để hỗ trợ cả